# BirdCLEF 2026 — ONNX Perch + CNN Blend

Full pipeline: ONNX Perch (7.98x faster than TF SavedModel) + probes + ProtoSSM + 1-fold CNN blend.
Target: beat 0.912 with Perch+CNN blend that fits in 90-min budget.


## Cell 1 — Install ONNX Runtime (offline)

Install from prebuilt wheels in `aldisued/onnxruntime-121-cp312-linux` dataset.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = ""  # CPU-only

# Install onnxruntime from offline wheels dataset
_WHEELS_DIR = None
for _wd_candidate in [
    Path("/kaggle/input/onnxruntime-121-cp312-linux"),
    Path("/kaggle/input/datasets/aldisued/onnxruntime-121-cp312-linux"),
]:
    if _wd_candidate.exists():
        _WHEELS_DIR = _wd_candidate
        break

if _WHEELS_DIR is not None:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--find-links",
            str(_WHEELS_DIR),
            "onnxruntime",
        ],
        check=True,
        capture_output=True,
    )
    print("Installed onnxruntime from local wheels.")
else:
    raise RuntimeError(
        "onnxruntime wheels not found! Add 'aldisued/onnxruntime-121-cp312-linux' to dataset_sources."
    )

import onnxruntime as ort

print(f"onnxruntime {ort.__version__}")

import numpy as np

## Cell 2 — Libraries

In [ ]:
import gc
import json
import os
import platform
import random
import re
import sys
import warnings
from pathlib import Path

import pandas as pd
import soundfile as sf
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # force CPU
tf.experimental.numpy.experimental_enable_numpy_behavior()

print("Python     :", sys.version)
print("Platform   :", platform.system(), platform.release())
print("TensorFlow :", tf.__version__)
print("NumPy      :", np.__version__)
print("Pandas     :", pd.__version__)

## Cell 3 — Config & Paths

In [ ]:
class Settings:
    # Mode ------------------------------------------------------------------
    # 'submit' : uses frozen hyperparameters, skips grid search
    # 'train'  : runs probe parameter grid search and prints OOF diagnostics
    MODE = "submit"
    SEED = 42

    # Competition paths -----------------------------------------------------
    _KAGGLE_BASE = Path("/kaggle/input/competitions/birdclef-2026")
    _LOCAL_BASE = Path("../dataset")
    BASE = _KAGGLE_BASE if _KAGGLE_BASE.exists() else _LOCAL_BASE

    # Perch v2 SavedModel ---------------------------------------------------
    MODEL_DIR = Path(
        "/kaggle/input/models/google/bird-vocalization-classifier"
        "/tensorflow2/perch_v2_cpu/1"
    )

    # Perch cache (V1 format: full_perch_meta.parquet + full_perch_arrays.npz)
    # Keys expected: scores_full_raw, emb_full
    _CACHE_CANDIDATES = [
        Path("/kaggle/input/datasets/jaejohn/perch-meta"),  # attached Kaggle dataset
        Path("../input/perch-meta"),  # local dev
        Path("/kaggle/working/cache"),  # freshly generated this session
    ]
    CACHE_DIR = next(
        (
            d
            for d in _CACHE_CANDIDATES
            if (d / "full_perch_meta.parquet").exists()
            and (d / "full_perch_arrays.npz").exists()
        ),
        None,
    )
    CACHE_EXISTS = CACHE_DIR is not None
    WORK_CACHE_DIR = Path("/kaggle/working/cache")

    # Audio -----------------------------------------------------------------
    SR = 32_000
    WINDOW_SEC = 5
    WINDOW_SAMPLES = SR * WINDOW_SEC  # 160,000 samples
    FILE_SAMPLES = 60 * SR  # 1,920,000 samples
    N_WINDOWS = 12  # 5-second windows per 60-second file
    BATCH_FILES = 16
    DRYRUN_N_FILES = 20  # dry-run when hidden test is not mounted

    # Prior fusion (frozen from OOF tuning) ---------------------------------
    LAMBDA_EVENT = 0.45  # V18: was 0.4
    LAMBDA_TEXTURE = 1.1  # V18: was 1.0
    LAMBDA_PROXY_TEXTURE = 0.9  # V18: was 0.8

    # Temporal smoothing — 5-way acoustic archetypes
    # Archetypes: amphibia (continuous chorus), insecta (texture),
    #   aves_direct (clear events), aves_uncertain (proxy/unmapped birds),
    #   rare (Mammalia/Reptilia — infrequent calls)
    SMOOTH_AMPHIBIA_ALPHA = 0.45  # strong smoothing — continuous background chorus
    SMOOTH_INSECTA_ALPHA = 0.35  # strong smoothing — texture
    SMOOTH_AVES_UNCERTAIN_ALPHA = (
        0.00  # no smoothing — event birds benefit from sharp peaks (OOF-confirmed)
    )
    SMOOTH_AVES_DIRECT_ALPHA = 0.00  # no smoothing — confirmed by OOF grid search
    SMOOTH_RARE_ALPHA = 0.05  # minimal — Mammalia/Reptilia, rare transients

    # Embedding probe (frozen from OOF tuning) ------------------------------
    PROBE_PCA_DIM = 128
    PROBE_MIN_POS = 5
    PROBE_C = 0.75
    PROBE_ALPHA = 0.45

    # MLP probe config
    PROBE_MLP_HIDDEN = (64,)  # 1 hidden layer of 64 units
    PROBE_MLP_ALPHA = 0.01  # L2 regularisation strength
    PROBE_MLP_MAX_ITER = 300


CFG = Settings()
CFG.WORK_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"MODE         : {CFG.MODE}")
print(f"BASE         : {CFG.BASE}")
print(f"MODEL_DIR    : {CFG.MODEL_DIR}")
print(f"CACHE_EXISTS : {CFG.CACHE_EXISTS}")
print(f"CACHE_DIR    : {CFG.CACHE_DIR}")

## Cell 4 — Seed

In [ ]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


seed_everything(CFG.SEED)

## Cell 5 — Load Taxonomy & Build Label Matrix

Load the four metadata files, strip duplicates from soundscape labels, build the multi-hot label matrix `Y_SC`, and identify the 59 fully-labeled soundscape files (all 12 windows annotated).

In [ ]:
taxonomy = pd.read_csv(CFG.BASE / "taxonomy.csv")
train_meta = pd.read_csv(CFG.BASE / "train.csv")
soundscape_raw = pd.read_csv(CFG.BASE / "train_soundscapes_labels.csv")
sample_sub = pd.read_csv(CFG.BASE / "sample_submission.csv")

soundscape_lbls = soundscape_raw.drop_duplicates().reset_index(drop=True)

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}

print(f"taxonomy species     : {len(taxonomy)}")
print(f"train recordings     : {len(train_meta):,}")
print(
    f"soundscape rows      : {len(soundscape_lbls):,} unique "
    f"(dropped {len(soundscape_raw) - len(soundscape_lbls):,} duplicates)"
)
print(f"submission classes   : {N_CLASSES}")

## Cell 6 — Parse Labels & Build Soundscape Metadata

In [ ]:
FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")


def parse_labels(x):
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split(";") if t.strip()]


def union_labels(series):
    return sorted(set(lbl for x in series for lbl in parse_labels(x)))


def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m:
        return {"site": None, "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}


# Aggregate labels per 5-second window and attach metadata
sc_clean = (
    soundscape_lbls.groupby(["filename", "start", "end"])["primary_label"]
    .apply(union_labels)
    .reset_index(name="label_list")
)
sc_clean["end_sec"] = pd.to_timedelta(sc_clean["end"]).dt.total_seconds().astype(int)
sc_clean["row_id"] = (
    sc_clean["filename"].str.replace(".ogg", "", regex=False)
    + "_"
    + sc_clean["end_sec"].astype(str)
)
meta_cols = sc_clean["filename"].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, meta_cols], axis=1)

# Identify fully-labeled files (all 12 windows annotated)
wpf = sc_clean.groupby("filename").size()
full_files = sorted(wpf[wpf == CFG.N_WINDOWS].index.tolist())
sc_clean["file_fully_labeled"] = sc_clean["filename"].isin(full_files)

# Multi-hot label matrix for all soundscape rows
Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)
for i, labels in enumerate(sc_clean["label_list"]):
    for lbl in labels:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1

# Ordered truth for fully-labeled files
full_truth = (
    sc_clean[sc_clean["file_fully_labeled"]]
    .sort_values(["filename", "end_sec"])
    .reset_index(drop=False)
)
Y_FULL_TRUTH = Y_SC[full_truth["index"].to_numpy()]

print(f"Fully-labeled files : {len(full_files)}")
print(f"Trusted windows     : {len(full_truth)}")
print(f"Active classes      : {int((Y_FULL_TRUTH.sum(axis=0) > 0).sum())}")

## Cell 7 — Load ONNX Perch Model & Map Labels

Load the ONNX-converted Perch v2 model and map its 14,795-class vocab to our 234 target species.

In [ ]:
import time

# ── Load ONNX Perch model ─────────────────────────────────────────────────────
_ONNX_CANDIDATES = [
    Path(
        "/kaggle/input/datasets/yuriygreben/birdclef26-perch-onnx/perch_v2_no_dft.onnx"
    ),
    Path("/kaggle/input/birdclef26-perch-onnx/perch_v2_no_dft.onnx"),
]
_onnx_path = next((p for p in _ONNX_CANDIDATES if p.exists()), None)
if _onnx_path is None:
    raise FileNotFoundError("Perch ONNX model not found")

print(f"Loading ONNX model from {_onnx_path} ...")
t0 = time.time()
_sess_opts = ort.SessionOptions()
_sess_opts.intra_op_num_threads = 2
_sess_opts.inter_op_num_threads = 2
_sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
ort_session = ort.InferenceSession(str(_onnx_path), _sess_opts)

_input_name = ort_session.get_inputs()[0].name  # "inputs"
_out_names = [o.name for o in ort_session.get_outputs()]
print(f"ONNX outputs: {_out_names}")
_emb_idx = next(i for i, n in enumerate(_out_names) if n == "embedding")
_label_idx = next(i for i, n in enumerate(_out_names) if n == "label")
print(f"Model loaded in {time.time() - t0:.1f}s")

# ── Label vocabulary (read from TF model directory ─ labels.csv only, no TF) ─
bc_labels = (
    pd.read_csv(CFG.MODEL_DIR / "assets" / "labels.csv")
    .reset_index()
    .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
)
NO_LABEL_INDEX = len(bc_labels)

taxonomy_ = taxonomy.copy()
taxonomy_["scientific_name"] = taxonomy_["scientific_name"].astype(str)
mapping = taxonomy_.merge(
    bc_labels[["scientific_name", "bc_index"]], on="scientific_name", how="left"
)
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL_INDEX).astype(int)
label_to_bc = mapping.set_index("primary_label")["bc_index"]

BC_INDICES = np.array([int(label_to_bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)

MAPPED_MASK = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
UNMAPPED_POS = np.where(~MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

print(f"Mapped   : {MAPPED_MASK.sum()} / {N_CLASSES}")
print(f"Unmapped : {(~MAPPED_MASK).sum()}")

## Cell 8 — Class Type Indices

Species are split into texture classes (Amphibia, Insecta) and event classes (everything else). This drives different prior fusion lambdas: frogs and insects are location-determined recurring textures, whereas birds are sparse acoustic events.

In [ ]:
CLASS_NAME_MAP = taxonomy_.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA = {"Amphibia", "Insecta"}
ACTIVE_CLASSES = [PRIMARY_LABELS[i] for i in np.where(Y_SC.sum(axis=0) > 0)[0]]

idx_active_texture = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) in TEXTURE_TAXA],
    dtype=np.int32,
)
idx_active_event = np.array(
    [
        label_to_idx[c]
        for c in ACTIVE_CLASSES
        if CLASS_NAME_MAP.get(c) not in TEXTURE_TAXA
    ],
    dtype=np.int32,
)

idx_mapped_active_texture = idx_active_texture[MAPPED_MASK[idx_active_texture]]
idx_mapped_active_event = idx_active_event[MAPPED_MASK[idx_active_event]]
idx_unmapped_active_texture = idx_active_texture[~MAPPED_MASK[idx_active_texture]]
idx_unmapped_active_event = idx_active_event[~MAPPED_MASK[idx_active_event]]
idx_unmapped_inactive = np.array(
    [i for i in UNMAPPED_POS if PRIMARY_LABELS[i] not in ACTIVE_CLASSES], dtype=np.int32
)

# 5-way smoothing archetypes (used only for temporal smoothing in fuse_scores)
_RARE_TAXA = {"Mammalia", "Reptilia"}
idx_smooth_amphibia = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) == "Amphibia"],
    dtype=np.int32,
)
idx_smooth_insecta = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) == "Insecta"],
    dtype=np.int32,
)
idx_smooth_rare = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) in _RARE_TAXA],
    dtype=np.int32,
)
idx_smooth_aves_direct = np.array(
    [
        label_to_idx[c]
        for c in ACTIVE_CLASSES
        if CLASS_NAME_MAP.get(c) == "Aves" and MAPPED_MASK[label_to_idx[c]]
    ],
    dtype=np.int32,
)
idx_smooth_aves_uncertain = np.array(
    [
        label_to_idx[c]
        for c in ACTIVE_CLASSES
        if CLASS_NAME_MAP.get(c) == "Aves" and not MAPPED_MASK[label_to_idx[c]]
    ],
    dtype=np.int32,
)

print(f"Active texture classes     : {len(idx_active_texture)}")
print(f"Active event classes       : {len(idx_active_event)}")
print(f"Unmapped inactive          : {len(idx_unmapped_inactive)}")
print(f"Smooth — Amphibia          : {len(idx_smooth_amphibia)}")
print(f"Smooth — Insecta           : {len(idx_smooth_insecta)}")
print(f"Smooth — Aves direct       : {len(idx_smooth_aves_direct)}")
print(f"Smooth — Aves uncertain    : {len(idx_smooth_aves_uncertain)}")
print(f"Smooth — Rare (Mam/Rep)    : {len(idx_smooth_rare)}")

## Cell 9 — Genus Proxies for Unmapped Species

For unmapped species (not insect sonotypes), Perch's label list is searched for any class sharing the same genus. The maximum logit across genus matches serves as a proxy score — better than leaving those columns at zero.

In [ ]:
unmapped_df = mapping[mapping["bc_index"] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[
    ~unmapped_df["primary_label"].astype(str).str.contains("son", na=False)
].copy()

proxy_map = {}
for _, row in unmapped_non_sonotype.iterrows():
    genus = str(row["scientific_name"]).split()[0]
    hits = bc_labels[
        bc_labels["scientific_name"].str.match(rf"^{re.escape(genus)}\s", na=False)
    ]
    if len(hits) > 0:
        proxy_map[str(row["primary_label"])] = hits["bc_index"].astype(int).tolist()

# Use genus proxy for ALL unmapped non-sonotype classes (including Aves not in training soundscapes)
# V18: extended from ACTIVE_LABELS filter to all proxy_map entries
SELECTED_PROXY_TARGETS = sorted(proxy_map.keys())
selected_proxy_pos = np.array(
    [label_to_idx[c] for c in SELECTED_PROXY_TARGETS], dtype=np.int32
)
selected_proxy_pos_to_bc = {
    label_to_idx[t]: np.array(proxy_map[t], dtype=np.int32)
    for t in SELECTED_PROXY_TARGETS
}

# Break down by class type for logging
proxy_by_class = {}
for t in SELECTED_PROXY_TARGETS:
    cn = CLASS_NAME_MAP.get(t, "Unknown")
    proxy_by_class.setdefault(cn, []).append(t)
for cn, targets in sorted(proxy_by_class.items()):
    print(f"  Proxy [{cn}]: {len(targets)} targets — {targets}")

idx_selected_proxy_active_texture = np.intersect1d(
    selected_proxy_pos, idx_active_texture
)
idx_selected_prioronly_active_texture = np.setdiff1d(
    idx_unmapped_active_texture, selected_proxy_pos
)
idx_selected_prioronly_active_event = np.setdiff1d(
    idx_unmapped_active_event, selected_proxy_pos
)

print(f"Total proxy targets  : {len(SELECTED_PROXY_TARGETS)}")
print(f"Proxy texture classes: {len(idx_selected_proxy_active_texture)}")
print(f"Prior-only texture   : {len(idx_selected_prioronly_active_texture)}")
print(f"Prior-only event     : {len(idx_selected_prioronly_active_event)}")

## Cell 10 — Audio & Perch Inference Helpers

In [ ]:
def read_soundscape_60s(path):
    """Load a 60-second soundscape, return mono float32 padded/truncated to FILE_SAMPLES."""
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < CFG.FILE_SAMPLES:
        y = np.pad(y, (0, CFG.FILE_SAMPLES - len(y)))
    return y[: CFG.FILE_SAMPLES]


def infer_perch_batch(paths, verbose=True):
    """
    Run Perch on a list of 60-second .ogg files.
    Returns (meta_df, scores, embeddings).
      meta_df    : DataFrame with row_id, filename, site, hour_utc
      scores     : (n_files * N_WINDOWS, N_CLASSES) float32 — competition-class logits
      embeddings : (n_files * N_WINDOWS, 1536)      float32
    """
    paths = [Path(p) for p in paths]
    n_files = len(paths)
    n_rows = n_files * CFG.N_WINDOWS

    row_ids = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites = np.empty(n_rows, dtype=object)
    hours = np.empty(n_rows, dtype=np.int16)
    scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embeddings = np.zeros((n_rows, 1536), dtype=np.float32)

    write_row = 0
    itr = tqdm(range(0, n_files, CFG.BATCH_FILES), desc="Perch", disable=not verbose)

    for start in itr:
        batch = paths[start : start + CFG.BATCH_FILES]
        bn = len(batch)
        x = np.empty((bn * CFG.N_WINDOWS, CFG.WINDOW_SAMPLES), dtype=np.float32)
        bstart = write_row

        for bi, path in enumerate(batch):
            audio = read_soundscape_60s(path)
            x[bi * CFG.N_WINDOWS : (bi + 1) * CFG.N_WINDOWS] = audio.reshape(
                CFG.N_WINDOWS, CFG.WINDOW_SAMPLES
            )
            meta = parse_soundscape_filename(path.name)
            row_ids[write_row : write_row + CFG.N_WINDOWS] = [
                f"{path.stem}_{t}" for t in range(5, 65, 5)
            ]
            filenames[write_row : write_row + CFG.N_WINDOWS] = path.name
            sites[write_row : write_row + CFG.N_WINDOWS] = meta["site"]
            hours[write_row : write_row + CFG.N_WINDOWS] = meta["hour_utc"]
            write_row += CFG.N_WINDOWS

        ort_out = ort_session.run(
            [_out_names[_emb_idx], _out_names[_label_idx]], {_input_name: x}
        )
        emb = ort_out[0].astype(np.float32)  # (n, 1536)
        logits = ort_out[1].astype(np.float32)  # (n, 14795)

        scores[bstart:write_row, MAPPED_POS] = logits[
            : write_row - bstart, MAPPED_BC_INDICES
        ]
        embeddings[bstart:write_row] = emb

        # Genus proxy: take max logit over matching Perch classes
        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            scores[bstart:write_row, pos] = logits[
                : write_row - bstart, bc_idx_arr
            ].max(axis=1)

        del x, ort_out, logits, emb
        gc.collect()

    meta_df = pd.DataFrame(
        {
            "row_id": row_ids,
            "filename": filenames,
            "site": sites,
            "hour_utc": hours,
        }
    )
    return meta_df, scores, embeddings

## Cell 11 — Load Training Soundscape Cache

If the `jaejohn/perch-meta` cache (708 rows × 234 classes, 708 × 1536 embeddings) is attached, load it directly. Otherwise, run Perch on the 59 fully-labeled training soundscapes and save to `/kaggle/working/cache/`.

In [ ]:
if CFG.CACHE_EXISTS:
    print(f"Loading Perch cache from: {CFG.CACHE_DIR}")
    meta_full = pd.read_parquet(CFG.CACHE_DIR / "full_perch_meta.parquet")
    arr = np.load(CFG.CACHE_DIR / "full_perch_arrays.npz")
    scores_full_raw = arr["scores_full_raw"].astype(np.float32)
    emb_full = arr["emb_full"].astype(np.float32)
else:
    print("No cache found. Running Perch on fully-labeled training soundscapes...")
    full_paths = [CFG.BASE / "train_soundscapes" / fn for fn in full_files]
    meta_full, scores_full_raw, emb_full = infer_perch_batch(full_paths)
    meta_full.to_parquet(CFG.WORK_CACHE_DIR / "full_perch_meta.parquet", index=False)
    np.savez_compressed(
        CFG.WORK_CACHE_DIR / "full_perch_arrays.npz",
        scores_full_raw=scores_full_raw,
        emb_full=emb_full,
    )
    print(f"Cache saved to {CFG.WORK_CACHE_DIR}")

# Align ground truth to cache row order
full_truth_aligned = (
    full_truth.set_index("row_id").loc[meta_full["row_id"]].reset_index(drop=False)
)
Y_FULL = Y_SC[full_truth_aligned["index"].to_numpy()]

print(f"scores_full_raw : {scores_full_raw.shape}  {scores_full_raw.dtype}")
print(f"emb_full        : {emb_full.shape}  {emb_full.dtype}")
print(f"Y_FULL          : {Y_FULL.shape}")

## Cell 12 — Bayesian Site × Hour Prior Tables

Build prior prevalence tables from soundscape metadata: global, site, hour, and site×hour joint priors with Dirichlet shrinkage. Prior logits are fused with raw Perch logits using class-type-dependent lambdas.

In [ ]:
def fit_prior_tables(prior_df, Y_prior):
    """Fit site / hour / site-hour prior tables with Dirichlet shrinkage toward global mean."""
    prior_df = prior_df.reset_index(drop=True)
    global_p = Y_prior.mean(axis=0).astype(np.float32)

    site_keys = sorted(prior_df["site"].dropna().astype(str).unique())
    hour_keys = sorted(prior_df["hour_utc"].dropna().astype(int).unique())

    site_to_i, site_n, site_p = {}, [], []
    for s in site_keys:
        mask = prior_df["site"].astype(str).values == s
        site_to_i[s] = len(site_n)
        site_n.append(mask.sum())
        site_p.append(Y_prior[mask].mean(axis=0))
    site_n = np.array(site_n, dtype=np.float32)
    site_p = (
        np.stack(site_p).astype(np.float32)
        if site_p
        else np.zeros((0, Y_prior.shape[1]), np.float32)
    )

    hour_to_i, hour_n, hour_p = {}, [], []
    for h in hour_keys:
        mask = prior_df["hour_utc"].astype(int).values == h
        hour_to_i[h] = len(hour_n)
        hour_n.append(mask.sum())
        hour_p.append(Y_prior[mask].mean(axis=0))
    hour_n = np.array(hour_n, dtype=np.float32)
    hour_p = (
        np.stack(hour_p).astype(np.float32)
        if hour_p
        else np.zeros((0, Y_prior.shape[1]), np.float32)
    )

    sh_to_i, sh_n_list, sh_p_list = {}, [], []
    for (s, h), idx in prior_df.groupby(["site", "hour_utc"]).groups.items():
        sh_to_i[(str(s), int(h))] = len(sh_n_list)
        idx = np.array(list(idx))
        sh_n_list.append(len(idx))
        sh_p_list.append(Y_prior[idx].mean(axis=0))
    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = (
        np.stack(sh_p_list).astype(np.float32)
        if sh_p_list
        else np.zeros((0, Y_prior.shape[1]), np.float32)
    )

    return dict(
        global_p=global_p,
        site_to_i=site_to_i,
        site_n=site_n,
        site_p=site_p,
        hour_to_i=hour_to_i,
        hour_n=hour_n,
        hour_p=hour_p,
        sh_to_i=sh_to_i,
        sh_n=sh_n,
        sh_p=sh_p,
    )


def prior_logits(sites, hours, tables, eps=1e-4):
    """Compute prior logits for a batch of (site, hour) pairs using Dirichlet shrinkage."""
    n = len(sites)
    p = np.repeat(tables["global_p"][None, :], n, axis=0).astype(np.float32, copy=True)

    si = np.fromiter((tables["site_to_i"].get(str(s), -1) for s in sites), np.int32, n)
    hi = np.fromiter(
        (tables["hour_to_i"].get(int(h), -1) if int(h) >= 0 else -1 for h in hours),
        np.int32,
        n,
    )
    shi = np.fromiter(
        (
            tables["sh_to_i"].get((str(s), int(h)), -1) if int(h) >= 0 else -1
            for s, h in zip(sites, hours)
        ),
        np.int32,
        n,
    )

    valid = hi >= 0
    if valid.any():
        nh = tables["hour_n"][hi[valid]][:, None]
        p[valid] = (
            nh / (nh + 8.0) * tables["hour_p"][hi[valid]]
            + (1.0 - nh / (nh + 8.0)) * p[valid]
        )

    valid = si >= 0
    if valid.any():
        ns = tables["site_n"][si[valid]][:, None]
        p[valid] = (
            ns / (ns + 8.0) * tables["site_p"][si[valid]]
            + (1.0 - ns / (ns + 8.0)) * p[valid]
        )

    valid = shi >= 0
    if valid.any():
        nsh = tables["sh_n"][shi[valid]][:, None]
        p[valid] = (
            nsh / (nsh + 4.0) * tables["sh_p"][shi[valid]]
            + (1.0 - nsh / (nsh + 4.0)) * p[valid]
        )

    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32)


def smooth_cols(scores, cols, alpha=0.35):
    """Temporal smoothing: blend each window with its neighbours within the same 60-second file."""
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s = scores.copy()
    view = s.reshape(-1, CFG.N_WINDOWS, s.shape[1])
    x = view[:, :, cols]
    prev = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    nxt = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev + nxt)
    return s


def fuse_scores(base, sites, hours, tables):
    """Fuse raw Perch logits with Bayesian prior logits using class-type-specific lambdas."""
    scores = base.copy()
    prior = prior_logits(sites, hours, tables)

    if len(idx_mapped_active_event):
        scores[:, idx_mapped_active_event] += (
            CFG.LAMBDA_EVENT * prior[:, idx_mapped_active_event]
        )
    if len(idx_mapped_active_texture):
        scores[:, idx_mapped_active_texture] += (
            CFG.LAMBDA_TEXTURE * prior[:, idx_mapped_active_texture]
        )
    if len(idx_selected_proxy_active_texture):
        scores[:, idx_selected_proxy_active_texture] += (
            CFG.LAMBDA_PROXY_TEXTURE * prior[:, idx_selected_proxy_active_texture]
        )
    if len(idx_selected_prioronly_active_event):
        scores[:, idx_selected_prioronly_active_event] = (
            CFG.LAMBDA_EVENT * prior[:, idx_selected_prioronly_active_event]
        )
    if len(idx_selected_prioronly_active_texture):
        scores[:, idx_selected_prioronly_active_texture] = (
            CFG.LAMBDA_TEXTURE * prior[:, idx_selected_prioronly_active_texture]
        )
    if len(idx_unmapped_inactive):
        scores[:, idx_unmapped_inactive] = -8.0

    scores = smooth_cols(scores, idx_smooth_amphibia, alpha=CFG.SMOOTH_AMPHIBIA_ALPHA)
    scores = smooth_cols(scores, idx_smooth_insecta, alpha=CFG.SMOOTH_INSECTA_ALPHA)
    scores = smooth_cols(
        scores, idx_smooth_aves_direct, alpha=CFG.SMOOTH_AVES_DIRECT_ALPHA
    )
    scores = smooth_cols(
        scores, idx_smooth_aves_uncertain, alpha=CFG.SMOOTH_AVES_UNCERTAIN_ALPHA
    )
    scores = smooth_cols(scores, idx_smooth_rare, alpha=CFG.SMOOTH_RARE_ALPHA)
    return scores.astype(np.float32), prior


print("Prior fusion functions defined.")

## Cell 13 — OOF Fused Scores (for Probe Training Features)

Apply `GroupKFold(5)` by site to produce out-of-fold prior-fused scores. This prevents data leakage when training the embedding probes, since the probe features include the fused score.

In [ ]:
def macro_auc(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")


gkf = GroupKFold(n_splits=5)
groups = meta_full["site"].to_numpy()

oof_base = np.zeros_like(scores_full_raw, dtype=np.float32)
oof_prior = np.zeros_like(scores_full_raw, dtype=np.float32)

for _, va_idx in tqdm(
    list(gkf.split(scores_full_raw, groups=groups)), desc="OOF folds"
):
    va_idx = np.sort(va_idx)
    val_sites = set(meta_full.iloc[va_idx]["site"].tolist())
    prior_m = ~sc_clean["site"].isin(val_sites).values
    tables = fit_prior_tables(
        sc_clean.loc[prior_m].reset_index(drop=True), Y_SC[prior_m]
    )
    oof_base[va_idx], oof_prior[va_idx] = fuse_scores(
        scores_full_raw[va_idx],
        meta_full.iloc[va_idx]["site"].to_numpy(),
        meta_full.iloc[va_idx]["hour_utc"].to_numpy(),
        tables,
    )

print(f"OOF baseline AUC (prior fusion only): {macro_auc(Y_FULL, oof_base):.6f}")

## Cell 14 — PCA on Embeddings & Prototype Construction

StandardScaler → PCA(32 components) on the 1536-dim embeddings. Also build per-class prototype vectors (mean of positive-example embeddings in PCA space) and family-level index maps used as features for the probes.

In [ ]:
emb_scaler = StandardScaler()
emb_scaled = emb_scaler.fit_transform(emb_full)

n_comp = min(CFG.PROBE_PCA_DIM, emb_scaled.shape[0] - 1, emb_scaled.shape[1])
emb_pca = PCA(n_components=n_comp)
Z_FULL = emb_pca.fit_transform(emb_scaled).astype(np.float32)

print(f"PCA components  : {n_comp}")
print(f"Explained var   : {emb_pca.explained_variance_ratio_.sum():.4f}")

# Class prototypes in PCA space: mean(Z[Y[:,c]==1]) for each class with enough positives
CLASS_PROTOTYPES = {}
for ci in range(N_CLASSES):
    pos_mask = Y_FULL[:, ci] == 1
    if pos_mask.sum() >= CFG.PROBE_MIN_POS:
        CLASS_PROTOTYPES[ci] = Z_FULL[pos_mask].mean(axis=0)
print(f"Class prototypes computed: {len(CLASS_PROTOTYPES)} classes")

# Family-level index mapping (used to compute family mean score as a feature)
FAMILY_MAP = taxonomy_.set_index("primary_label")["class_name"].to_dict()
FAMILY_GROUPS = {}
for ci, label in enumerate(PRIMARY_LABELS):
    family = FAMILY_MAP.get(label, "Unknown")
    FAMILY_GROUPS.setdefault(family, []).append(ci)
FAMILY_IDX_MAP = {
    fam: np.array(idxs, dtype=np.int32) for fam, idxs in FAMILY_GROUPS.items()
}
CLASS_FAMILY = {
    ci: FAMILY_MAP.get(label, "Unknown") for ci, label in enumerate(PRIMARY_LABELS)
}
print(f"Family groups: {len(FAMILY_GROUPS)} — {sorted(FAMILY_GROUPS.keys())}")

## Cell 15 — Feature Helpers for Probes

43-feature vector per window per class:
- 32 PCA components
- raw logit, prior logit, fused (base) logit
- sequential features: prev, next, mean, max, min, range (6)
- prototype cosine similarity (1)
- family-level mean score (1)

In [ ]:
def seq_features_1d(v):
    """Compute sequential context features for a 1-D score vector shaped (n_files*N_WINDOWS,)."""
    x = v.reshape(-1, CFG.N_WINDOWS)
    prev = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    nxt = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
    mean_v = np.repeat(x.mean(1), CFG.N_WINDOWS)
    max_v = np.repeat(x.max(1), CFG.N_WINDOWS)
    min_v = np.repeat(x.min(1), CFG.N_WINDOWS)
    range_v = max_v - min_v
    return prev, nxt, mean_v, max_v, min_v, range_v


def cosine_sim_to_prototype(Z, prototype):
    """Cosine similarity between each row of Z and a single prototype vector."""
    Z_norm = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-8)
    p_norm = prototype / (np.linalg.norm(prototype) + 1e-8)
    return Z_norm @ p_norm  # (n,)


def build_class_features(
    Z, raw_col, prior_col, base_col, proto_sim_col=None, family_mean_col=None
):
    """
    Build the full feature matrix for one class.
    Shape: (n_windows, 32 + 3 + 6 [+ 1] [+ 1])  =  41-43 features.
    """
    p, n, m, mx, mn, rng = seq_features_1d(base_col)
    parts = [
        Z,
        raw_col[:, None],
        prior_col[:, None],
        base_col[:, None],
        p[:, None],
        n[:, None],
        m[:, None],
        mx[:, None],
        mn[:, None],
        rng[:, None],
    ]
    if proto_sim_col is not None:
        parts.append(proto_sim_col[:, None])
    if family_mean_col is not None:
        parts.append(family_mean_col[:, None])
    return np.concatenate(parts, axis=1).astype(np.float32)


print("Feature helpers defined.")

## Cell 16 — (Optional) Probe Parameter Grid Search

Only runs when `CFG.MODE == 'train'`. In submit mode the frozen params from Settings are used.

In [ ]:
if CFG.MODE == "train":
    param_grid = [
        {"pca_dim": 32, "min_pos": 8, "C": 0.25, "alpha": 0.40},
        {"pca_dim": 64, "min_pos": 8, "C": 0.25, "alpha": 0.40},
        {"pca_dim": 64, "min_pos": 8, "C": 0.25, "alpha": 0.50},
        {"pca_dim": 64, "min_pos": 8, "C": 0.50, "alpha": 0.40},
        {"pca_dim": 64, "min_pos": 12, "C": 0.25, "alpha": 0.40},
        {"pca_dim": 96, "min_pos": 8, "C": 0.25, "alpha": 0.40},
        {"pca_dim": 96, "min_pos": 8, "C": 0.50, "alpha": 0.40},
    ]
    rows = []
    for p in tqdm(param_grid, desc="Probe grid"):
        _scaler = StandardScaler()
        _pca = PCA(
            n_components=min(p["pca_dim"], emb_full.shape[0] - 1, emb_full.shape[1])
        )
        _Z = _pca.fit_transform(_scaler.fit_transform(emb_full)).astype(np.float32)
        _gkf = GroupKFold(n_splits=5)
        _groups = meta_full["site"].to_numpy()
        _oof = oof_base.copy()
        for _, va_idx in _gkf.split(scores_full_raw, groups=_groups):
            tr_idx = np.setdiff1d(np.arange(len(scores_full_raw)), va_idx)
            pos_cnt = Y_FULL[tr_idx].sum(axis=0)
            for ci in np.where(pos_cnt >= p["min_pos"])[0]:
                y_tr = Y_FULL[tr_idx, ci]
                if y_tr.sum() == 0 or y_tr.sum() == len(y_tr):
                    continue
                X_tr = build_class_features(
                    _Z[tr_idx],
                    scores_full_raw[tr_idx, ci],
                    oof_prior[tr_idx, ci],
                    oof_base[tr_idx, ci],
                )
                X_va = build_class_features(
                    _Z[va_idx],
                    scores_full_raw[va_idx, ci],
                    oof_prior[va_idx, ci],
                    oof_base[va_idx, ci],
                )
                clf = LogisticRegression(
                    C=p["C"], max_iter=400, solver="liblinear", class_weight="balanced"
                )
                clf.fit(X_tr, y_tr)
                pred = clf.decision_function(X_va).astype(np.float32)
                _oof[va_idx, ci] = (1.0 - p["alpha"]) * oof_base[va_idx, ci] + p[
                    "alpha"
                ] * pred
        rows.append({**p, "oof_auc": macro_auc(Y_FULL, _oof)})
    grid_df = (
        pd.DataFrame(rows)
        .sort_values("oof_auc", ascending=False)
        .reset_index(drop=True)
    )
    print(grid_df.to_string(index=False))
    best = grid_df.iloc[0]
    print(
        f"\nBest: pca_dim={int(best.pca_dim)} min_pos={int(best.min_pos)} "
        f"C={best.C} alpha={best.alpha}  OOF AUC={best.oof_auc:.6f}"
    )
else:
    print(
        f"Using frozen probe params: pca_dim={CFG.PROBE_PCA_DIM} "
        f"min_pos={CFG.PROBE_MIN_POS} C={CFG.PROBE_C} alpha={CFG.PROBE_ALPHA}"
    )

## Cell 17 — Fit Per-Class MLP Probes on Full Training Data

In [ ]:
pos_counts = Y_FULL.sum(axis=0)
probe_idx = np.where(pos_counts >= CFG.PROBE_MIN_POS)[0].astype(np.int32)
probe_models = {}

for cls_idx in tqdm(probe_idx, desc="Training probes"):
    y = Y_FULL[:, cls_idx]
    if y.sum() == 0 or y.sum() == len(y):
        continue

    # Prototype cosine similarity feature
    proto_sim = None
    if cls_idx in CLASS_PROTOTYPES:
        proto_sim = cosine_sim_to_prototype(Z_FULL, CLASS_PROTOTYPES[cls_idx])

    # Family-level mean score feature (exclude current class to avoid leakage)
    family_name = CLASS_FAMILY.get(cls_idx, "Unknown")
    family_idxs = FAMILY_IDX_MAP.get(family_name, np.array([]))
    other_family = family_idxs[family_idxs != cls_idx]
    family_mean = (
        oof_base[:, other_family].mean(axis=1) if len(other_family) > 0 else None
    )

    X = build_class_features(
        Z_FULL,
        raw_col=scores_full_raw[:, cls_idx],
        prior_col=oof_prior[:, cls_idx],
        base_col=oof_base[:, cls_idx],
        proto_sim_col=proto_sim,
        family_mean_col=family_mean,
    )
    clf = LogisticRegression(
        C=CFG.PROBE_C, max_iter=400, solver="liblinear", class_weight="balanced"
    )
    clf.fit(X, y)
    probe_models[cls_idx] = clf

print(f"Probe models trained : {len(probe_models)} / {N_CLASSES} classes")

## Cell 18 — Perch Inference on Test Soundscapes

Fit final prior tables on all labeled data (no fold exclusion), then run the full pipeline on test soundscapes. Falls back to a dry-run on 20 training files when no test soundscapes are present (normal in Kaggle sandbox — real audio is injected only in scoring environment).

In [ ]:
# Final prior tables: fitted on all labeled data
final_tables = fit_prior_tables(sc_clean.reset_index(drop=True), Y_SC)

# Locate test files; fall back to training soundscapes for dry-run
test_paths = sorted((CFG.BASE / "test_soundscapes").glob("*.ogg"))
if len(test_paths) == 0:
    print(f"No test soundscapes found. Dry-run on {CFG.DRYRUN_N_FILES} train files.")
    test_paths = sorted((CFG.BASE / "train_soundscapes").glob("*.ogg"))[
        : CFG.DRYRUN_N_FILES
    ]
else:
    print(f"Test files : {len(test_paths)}")

meta_test, scores_test_raw, emb_test = infer_perch_batch(test_paths)

# Prior fusion on test
test_base, test_prior = fuse_scores(
    scores_test_raw,
    meta_test["site"].to_numpy(),
    meta_test["hour_utc"].to_numpy(),
    final_tables,
)

# PCA projection of test embeddings
Z_TEST = emb_pca.transform(emb_scaler.transform(emb_test)).astype(np.float32)

print(f"Z_TEST shape : {Z_TEST.shape}")

## Cell 19 — Apply Probes to Test Data

In [ ]:
final_scores = test_base.copy()

for cls_idx, clf in tqdm(probe_models.items(), desc="Applying probes"):
    # Same prototype + family features as training
    proto_sim = None
    if cls_idx in CLASS_PROTOTYPES:
        proto_sim = cosine_sim_to_prototype(Z_TEST, CLASS_PROTOTYPES[cls_idx])

    family_name = CLASS_FAMILY.get(cls_idx, "Unknown")
    family_idxs = FAMILY_IDX_MAP.get(family_name, np.array([]))
    other_family = family_idxs[family_idxs != cls_idx]
    family_mean = (
        test_base[:, other_family].mean(axis=1) if len(other_family) > 0 else None
    )

    X = build_class_features(
        Z_TEST,
        raw_col=scores_test_raw[:, cls_idx],
        prior_col=test_prior[:, cls_idx],
        base_col=test_base[:, cls_idx],
        proto_sim_col=proto_sim,
        family_mean_col=family_mean,
    )
    pred = clf.decision_function(X).astype(np.float32)
    final_scores[:, cls_idx] = (1.0 - CFG.PROBE_ALPHA) * test_base[
        :, cls_idx
    ] + CFG.PROBE_ALPHA * pred

print(f"final_scores : {final_scores.shape}")
print(f"Score range  : {final_scores.min():.3f} to {final_scores.max():.3f}")

## Cell 19b — XC-Trained Probes for Additional Species

Apply pre-trained XC probes (perch_probes_v2.pkl) for species not covered by soundscape probes. Uses a simple PCA-only feature vector (no temporal context) with lower alpha.

In [ ]:
import pickle

# XC probe dataset path
_XC_PROBE_CANDIDATES = [
    Path("/kaggle/input/datasets/aldisued/birdclef2026-perch-xc-probes"),
    Path("../input/perch-xc-probes"),
]
_XC_PROBE_DIR = next(
    (d for d in _XC_PROBE_CANDIDATES if (d / "perch_probes_v2.pkl").exists()), None
)

if _XC_PROBE_DIR is not None:
    with open(_XC_PROBE_DIR / "perch_probes_v2.pkl", "rb") as f:
        xc_probes_data = pickle.load(f)

    xc_scaler = xc_probes_data["scaler"]
    xc_pca = xc_probes_data["pca"]
    xc_probe_models = xc_probes_data["probes"]  # dict[str, LogReg | None]
    xc_include_logit = xc_probes_data.get("include_perch_logit", False)

    # Transform test embeddings using XC PCA (fitted on XC clip embeddings, not soundscape)
    Z_TEST_XC = xc_pca.transform(xc_scaler.transform(emb_test)).astype(np.float32)

    # Species already covered by soundscape probes (complex 43-dim features)
    soundscape_covered = set(probe_models.keys())  # cls_idx ints
    XC_PROBE_ALPHA = 0.30  # lower alpha: XC → soundscape domain shift

    n_xc_applied = 0
    for sp, clf in xc_probe_models.items():
        if clf is None:
            continue
        cls_idx = label_to_idx.get(sp)
        if cls_idx is None or cls_idx in soundscape_covered:
            continue  # already covered by soundscape probes
        # Features: PCA dims + optionally raw Perch score
        if xc_include_logit:
            feat = np.concatenate(
                [Z_TEST_XC, scores_test_raw[:, cls_idx : cls_idx + 1]], axis=1
            )
        else:
            feat = Z_TEST_XC
        pred = clf.decision_function(feat).astype(np.float32)
        final_scores[:, cls_idx] = (1.0 - XC_PROBE_ALPHA) * final_scores[
            :, cls_idx
        ] + XC_PROBE_ALPHA * pred
        n_xc_applied += 1

    print(f"XC probes applied  : {n_xc_applied} additional species")
    print(f"Soundscape probes  : {len(soundscape_covered)} species")
    print(
        f"Total with probes  : {len(soundscape_covered) + n_xc_applied} / {N_CLASSES}"
    )
else:
    print("XC probe dataset not attached — skipping XC probes")

## Cell 19c — ProtoSSM Temporal Model (artifact mode)

Load pre-trained ProtoSSM v1 checkpoint (if available) and blend its logits
with the probe-enhanced `final_scores`. ProtoSSM is a 4-layer bidirectional
Mamba-style SSM trained on 708 soundscape windows (59 files × 12 windows).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── ProtoSSM model classes (copied from train_protossm.py) ──────────────────
_D_INPUT = 1536
_D_MODEL = 320
_D_STATE = 32
_N_SSM_LAYERS = 4
_N_PROTOTYPES = 2
_CROSS_ATTN_HEADS = 8
_META_DIM = 24
_N_SITES = 20
_DROPOUT = 0.12
_N_CLASSES = 234

# ResidualSSM second pass (v2)
_D_RESIDUAL = 128
_D_STATE_RESIDUAL = 16
_DROPOUT_RESIDUAL = 0.20


class _SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state, dropout):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.in_proj = nn.Linear(d_model, d_model * 2)
        self.conv1d = nn.Conv1d(
            d_model, d_model, kernel_size=3, padding=1, groups=d_model
        )
        self.dt_proj = nn.Linear(d_model, d_model)
        self.B_proj = nn.Linear(d_model, d_state)
        self.C_proj = nn.Linear(d_model, d_state)
        A_log = torch.arange(1, d_state + 1, dtype=torch.float32).log()
        self.A_log = nn.Parameter(A_log)
        self.D = nn.Parameter(torch.ones(d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        T, d = x.shape
        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_ssm.unsqueeze(0).permute(0, 2, 1))
        x_conv = x_conv.permute(0, 2, 1).squeeze(0)
        x_ssm = F.silu(x_conv)
        dt = F.softplus(self.dt_proj(x_ssm))
        B = self.B_proj(x_ssm)
        C = self.C_proj(x_ssm)
        A = -torch.exp(self.A_log)
        dA = torch.exp(dt.unsqueeze(-1) * A)
        h = torch.zeros(d, self.d_state, device=x.device)
        ys = []
        for t in range(T):
            dBt = dt[t].unsqueeze(-1) * B[t]
            h = h * dA[t] + dBt
            ys.append((h * C[t]).sum(-1))
        y = torch.stack(ys, dim=0) + x_ssm * self.D
        return self.dropout(y * F.silu(z))


class _BidirectionalSSMBlock(nn.Module):
    def __init__(self, d_model, d_state, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.ssm_fwd = _SelectiveSSM(d_model, d_state, dropout)
        self.ssm_bwd = _SelectiveSSM(d_model, d_state, dropout)
        self.merge = nn.Linear(d_model * 2, d_model)
        self.merge_norm = nn.LayerNorm(d_model)

    def forward(self, x):
        residual = x
        x_norm = self.norm(x)
        fwd_out = self.ssm_fwd(x_norm)
        bwd_out = self.ssm_bwd(x_norm.flip(0)).flip(0)
        merged = self.merge(torch.cat([fwd_out, bwd_out], dim=-1))
        return self.merge_norm(merged + residual)


class _TemporalCrossAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x_norm = self.norm(x).unsqueeze(0)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        return residual + self.dropout(attn_out.squeeze(0))


class _ResidualSSM(nn.Module):
    """2-layer BiSSM on first-pass residuals for temporal correction (v2)."""

    def __init__(
        self,
        n_classes,
        d_residual=_D_RESIDUAL,
        d_state=_D_STATE_RESIDUAL,
        dropout=_DROPOUT_RESIDUAL,
    ):
        super().__init__()
        self.in_proj = nn.Sequential(
            nn.Linear(n_classes, d_residual),
            nn.LayerNorm(d_residual),
            nn.Dropout(dropout),
        )
        self.ssm1 = _BidirectionalSSMBlock(d_residual, d_state, dropout)
        self.ssm2 = _BidirectionalSSMBlock(d_residual, d_state, dropout)
        self.out_proj = nn.Linear(d_residual, n_classes)
        nn.init.zeros_(self.out_proj.weight)
        nn.init.zeros_(self.out_proj.bias)

    def forward(self, residual):
        h = self.in_proj(residual)
        h = self.ssm1(h)
        h = self.ssm2(h)
        return self.out_proj(h)


class _ProtoSSM(nn.Module):
    def __init__(
        self,
        d_input=_D_INPUT,
        d_model=_D_MODEL,
        d_state=_D_STATE,
        n_ssm_layers=_N_SSM_LAYERS,
        n_prototypes=_N_PROTOTYPES,
        cross_attn_heads=_CROSS_ATTN_HEADS,
        meta_dim=_META_DIM,
        n_sites=_N_SITES,
        dropout=_DROPOUT,
        n_classes=_N_CLASSES,
        n_tax_groups=5,
        has_residual_ssm=False,
    ):
        super().__init__()
        self.n_classes = n_classes
        self.n_prototypes = n_prototypes
        self.has_residual_ssm = has_residual_ssm
        self.site_embed = nn.Embedding(n_sites + 1, meta_dim // 2, padding_idx=0)
        self.hour_embed = nn.Embedding(24, meta_dim // 2)
        self.input_proj = nn.Sequential(
            nn.Linear(d_input + meta_dim, d_model),
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
        )
        self.ssm_layers = nn.ModuleList(
            [
                _BidirectionalSSMBlock(d_model, d_state, dropout)
                for _ in range(n_ssm_layers)
            ]
        )
        self.cross_attn = _TemporalCrossAttention(d_model, cross_attn_heads, dropout)
        self.prototypes = nn.Parameter(torch.randn(n_classes * n_prototypes, d_model))
        nn.init.xavier_uniform_(self.prototypes.view(n_classes * n_prototypes, d_model))
        self.proto_temp = nn.Parameter(torch.zeros(1))
        self.class_bias = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))
        self.tax_head = nn.Linear(d_model, n_tax_groups)
        self.dropout = nn.Dropout(dropout)
        if has_residual_ssm:
            self.residual_ssm = _ResidualSSM(n_classes)

    def forward(self, emb, perch_logits, site_idx, hour_idx):
        T = emb.shape[0]
        s_emb = self.site_embed(site_idx).expand(T, -1)
        h_emb = self.hour_embed(hour_idx).expand(T, -1)
        meta = torch.cat([s_emb, h_emb], dim=-1)
        x = self.input_proj(torch.cat([emb, meta], dim=-1))
        for layer in self.ssm_layers:
            x = layer(x)
        h = self.cross_attn(x)
        h_norm = F.normalize(h, dim=-1)
        p_norm = F.normalize(self.prototypes, dim=-1)
        sim = (h_norm @ p_norm.T).reshape(T, self.n_classes, self.n_prototypes)
        sim = sim.max(-1).values * F.softplus(self.proto_temp) + self.class_bias
        alpha = torch.sigmoid(self.fusion_alpha)
        logits = alpha * sim + (1 - alpha) * perch_logits
        if self.has_residual_ssm:
            residual = logits - perch_logits
            delta = self.residual_ssm(residual)
            logits = logits + delta
        return logits, self.tax_head(h.mean(0))


# ── Load checkpoint ──────────────────────────────────────────────────────────
PROTOSSM_WEIGHT = 0.5  # blend weight: 0=probe only, 1=ProtoSSM only
_PROTO_CANDIDATES = [
    Path("/kaggle/input/datasets/aldisued/birdclef2026-protossm-v2/protossm_v2.pt"),
    Path("/kaggle/input/birdclef2026-protossm-v2/protossm_v2.pt"),
    Path("/kaggle/input/datasets/aldisued/birdclef2026-protossm-v1/protossm_v1.pt"),
    Path("/kaggle/input/birdclef2026-protossm-v1/protossm_v1.pt"),
]
_proto_ckpt_path = next((p for p in _PROTO_CANDIDATES if p.exists()), None)

if _proto_ckpt_path is not None:
    print(f"Loading ProtoSSM from {_proto_ckpt_path}")
    _ckpt = torch.load(str(_proto_ckpt_path), map_location="cpu", weights_only=False)
    _cfg = _ckpt["config"]
    _site_to_idx = _ckpt["site_to_idx"]  # dict: site_str → int (1-based, 0=unknown)

    proto_model = _ProtoSSM(
        d_input=_cfg.get("d_input", _D_INPUT),
        d_model=_cfg.get("d_model", _D_MODEL),
        d_state=_cfg.get("d_state", _D_STATE),
        n_ssm_layers=_cfg.get("n_ssm_layers", _N_SSM_LAYERS),
        n_prototypes=_cfg.get("n_prototypes", _N_PROTOTYPES),
        cross_attn_heads=_cfg.get("cross_attn_heads", _CROSS_ATTN_HEADS),
        meta_dim=_cfg.get("meta_dim", _META_DIM),
        n_sites=_cfg.get("n_sites", _N_SITES),
        dropout=0.0,  # no dropout at inference
        n_classes=_cfg.get("n_classes", _N_CLASSES),
        n_tax_groups=len(_cfg.get("tax_group_names", [""] * 5)),
        has_residual_ssm=_cfg.get("has_residual_ssm", False),
    )
    proto_model.load_state_dict(_ckpt["model_state_dict"])
    proto_model.eval()
    _proto_ver = "v2+ResidualSSM" if _cfg.get("has_residual_ssm") else "v1"
    print(f"ProtoSSM loaded ({_proto_ver}). Sites: {sorted(_site_to_idx.keys())}")

    # ── Run ProtoSSM inference per soundscape file ───────────────────────────
    n_test_files = len(emb_test) // CFG.N_WINDOWS
    proto_logits_all = np.zeros_like(final_scores)

    # meta_test has one row per window — grab site/hour from first window of each file
    # meta_test index aligns with emb_test / scores_test_raw rows
    meta_test_rows = meta_test.reset_index(drop=True)

    with torch.no_grad():
        for fi in range(n_test_files):
            start = fi * CFG.N_WINDOWS
            end = start + CFG.N_WINDOWS

            emb_f = torch.tensor(emb_test[start:end], dtype=torch.float32)
            logits_f = torch.tensor(scores_test_raw[start:end], dtype=torch.float32)

            # Site / hour from filename metadata (first window of file)
            row = meta_test_rows.iloc[start]
            site_str = row.get("site") if hasattr(row, "get") else row["site"]
            hour_val = row.get("hour_utc") if hasattr(row, "get") else row["hour_utc"]

            site_i = _site_to_idx.get(site_str, 0) if site_str is not None else 0
            hour_i = int(hour_val) if (hour_val is not None and hour_val >= 0) else 0
            hour_i = max(0, min(23, hour_i))

            site_t = torch.tensor(site_i, dtype=torch.long)
            hour_t = torch.tensor(hour_i, dtype=torch.long)

            out_logits, _ = proto_model(emb_f, logits_f, site_t, hour_t)
            proto_logits_all[start:end] = out_logits.numpy()

    # Blend ProtoSSM logits with probe-enhanced logits
    final_scores = (
        PROTOSSM_WEIGHT * proto_logits_all + (1 - PROTOSSM_WEIGHT) * final_scores
    )
    print(
        f"ProtoSSM inference done. Blend={PROTOSSM_WEIGHT:.2f}×ProtoSSM + {1 - PROTOSSM_WEIGHT:.2f}×probes"
    )
    print(
        f"final_scores after blend: {final_scores.shape}, range [{final_scores.min():.3f}, {final_scores.max():.3f}]"
    )
else:
    print("ProtoSSM checkpoint not found — skipping (using probe scores only)")

## Cell 20 — Gaussian Temporal Smoothing & Build Submission

Apply a Gaussian kernel `[0.1, 0.2, 0.4, 0.2, 0.1]` along the time axis within each 60-second file. Convert logits to probabilities via sigmoid, then write `submission.csv`.

In [ ]:
def _gauss_conv(x, weights):
    """1D Gaussian convolution along axis 0 using numpy (replaces scipy.ndimage.convolve1d)."""
    pad = len(weights) // 2
    xp = np.pad(x, ((pad, pad), (0, 0)), mode="edge")
    out = np.zeros_like(x)
    for k, w in enumerate(weights):
        out += w * xp[k : k + len(x)]
    return out


def gauss_smooth_final(scores, weights=np.array([0.1, 0.2, 0.4, 0.2, 0.1])):
    """Apply Gaussian smoothing along the time axis within each 60-second file."""
    smoothed = scores.reshape(-1, CFG.N_WINDOWS, scores.shape[1]).copy()
    for i in range(smoothed.shape[0]):
        smoothed[i] = _gauss_conv(smoothed[i], weights)
    return smoothed.reshape(-1, scores.shape[1])


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


final_scores_smoothed = gauss_smooth_final(final_scores)

# --------------------------------------------------------------------------
# CNN blend: final_scores_smoothed already in logit space
# blend with CNN probs (converted to logit space) weighted by CNN_WEIGHT
# --------------------------------------------------------------------------
CNN_WEIGHT = 0.20  # 20% CNN, 80% Perch pipeline

if "cnn_rows" in dir() and cnn_rows:
    print(f"Blending {CNN_WEIGHT:.0%} CNN with {1 - CNN_WEIGHT:.0%} Perch pipeline ...")
    # Build CNN probs array aligned to meta_test row order (same as perch_probs)
    cnn_probs_arr = np.zeros((len(meta_test), cnn_n_species), dtype=np.float32)
    for ri, row_id in enumerate(meta_test["row_id"].values):
        if row_id in cnn_rows:
            cnn_probs_arr[ri] = cnn_rows[row_id]
    # Convert CNN probs to logits, then blend
    cnn_logits = np.log(
        np.clip(cnn_probs_arr, 1e-7, 1 - 1e-7)
        / np.clip(1 - cnn_probs_arr, 1e-7, 1 - 1e-7)
    ).astype(np.float32)
    # final_scores already blended (Perch logits in per-window space)
    # We blend the final per-window probs after smoothing
    perch_probs = 1.0 / (1.0 + np.exp(-final_scores_smoothed))
    blended_probs = (1 - CNN_WEIGHT) * perch_probs + CNN_WEIGHT * cnn_probs_arr
    probs_final = blended_probs.astype(np.float32)
    print(
        f"Blend complete. probs_final range: [{probs_final.min():.4f}, {probs_final.max():.4f}]"
    )
else:
    print("No CNN predictions found — using Perch pipeline only")
    probs_final = (1.0 / (1.0 + np.exp(-final_scores_smoothed))).astype(np.float32)

# Rank-aware post-processing: scale each species by file_max^0.4
# Applied to probs_final (CNN-blended or Perch-only)
_probs_3d = probs_final.reshape(-1, CFG.N_WINDOWS, N_CLASSES)
file_max = _probs_3d.max(axis=1)  # (n_files, N_CLASSES)
rank_scale = np.power(file_max, 0.4)  # (n_files, N_CLASSES)
_probs_3d = _probs_3d * rank_scale[:, np.newaxis, :]
probs_final = _probs_3d.reshape(-1, N_CLASSES).astype(np.float32)

submission = pd.DataFrame(probs_final, columns=PRIMARY_LABELS)
submission.insert(0, "row_id", meta_test["row_id"].values)
submission[PRIMARY_LABELS] = submission[PRIMARY_LABELS].astype(np.float32)

# Reindex to sample_submission row order; fill any missing rows with 0
submission = (
    submission.set_index("row_id")
    .reindex(sample_sub["row_id"])
    .fillna(0.0)
    .reset_index()
)
submission.rename(columns={"index": "row_id"}, inplace=True)

# Integrity checks
assert len(submission) == len(sample_sub), (
    f"Row count mismatch: {len(submission)} vs {len(sample_sub)}"
)
assert submission.columns.tolist() == ["row_id"] + PRIMARY_LABELS, (
    "Column order mismatch"
)
assert not submission.isna().any().any(), "NaNs detected in submission"
assert (submission[PRIMARY_LABELS] >= 0).all().all(), "Negative probabilities"
assert (submission[PRIMARY_LABELS] <= 1).all().all(), "Probabilities > 1"

submission.to_csv("submission.csv", index=False)
print("submission.csv saved")
print(f"Shape  : {submission.shape}")
submission.iloc[:3, :8]